# Configuration & Data Loading

In [1]:
from pathlib import Path

import pandas as pd


ROOT = Path("/kaggle/input/competitions/pokemon-tcg-ai-battle-challenge-strategy")

CARD_FILES = {
    "EN": ROOT / "EN_Card_Data.csv",
    "JP": ROOT / "JP_Card_Data.csv",
}


def load_card_database() -> dict[str, pd.DataFrame]:
    databases = {}

    for language, path in CARD_FILES.items():
        dataframe = pd.read_csv(path)
        dataframe.columns = dataframe.columns.str.strip()
        dataframe["Language"] = language
        databases[language] = dataframe

    return databases


def merge_card_database(databases: dict[str, pd.DataFrame]) -> pd.DataFrame:
    return pd.concat(databases.values(), ignore_index=True)


card_database = load_card_database()
cards = merge_card_database(card_database)

print(f"Total cards : {len(cards):,}")
print(f"Languages   : {', '.join(card_database.keys())}")

display(cards.head())

Total cards : 4,044
Languages   : EN, JP


,Card ID,Card Name,Expansion,Collection No.,Stage (Pokémon)/Type (Energy and Trainer),Rule,Category,Previous stage,HP,Type,...,カテゴリ,進化前,タイプ,弱点,抵抗力,にげる,ワザ名,コスト,ダメージ,効果の説明
0,1.0,Basic {G} Energy,SVE,1.0,Basic Energy,NaN,NaN,NaN,NaN,{G},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,Basic {R} Energy,SVE,2.0,Basic Energy,NaN,NaN,NaN,NaN,{R},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,Basic {W} Energy,SVE,3.0,Basic Energy,NaN,NaN,NaN,NaN,{W},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,Basic {L} Energy,SVE,4.0,Basic Energy,NaN,NaN,NaN,NaN,{L},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,Basic {P} Energy,SVE,5.0,Basic Energy,NaN,NaN,NaN,NaN,{P},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Card Database Exploration

In [2]:
from dataclasses import dataclass


@dataclass
class CardDatabaseSummary:
    total_cards: int
    pokemon: int
    trainer: int
    energy: int


def summarize_database(dataframe: pd.DataFrame) -> CardDatabaseSummary:
    type_column = next(
        column
        for column in dataframe.columns
        if "type" in column.lower()
    )

    types = dataframe[type_column].astype(str).str.lower()

    pokemon = types.str.contains("pokemon").sum()
    trainer = types.str.contains("trainer").sum()
    energy = types.str.contains("energy").sum()

    return CardDatabaseSummary(
        total_cards=len(dataframe),
        pokemon=pokemon,
        trainer=trainer,
        energy=energy,
    )


summary = summarize_database(cards)

overview = pd.DataFrame(
    {
        "Category": [
            "Total Cards",
            "Pokémon",
            "Trainer",
            "Energy",
        ],
        "Count": [
            summary.total_cards,
            summary.pokemon,
            summary.trainer,
            summary.energy,
        ],
    }
)

display(overview)

for language, dataframe in card_database.items():
    print(f"\n{language} Database")
    display(dataframe.head())

,Category,Count
0,Total Cards,4044
1,Pokémon,0
2,Trainer,0
3,Energy,20



EN Database


,Card ID,Card Name,Expansion,Collection No.,Stage (Pokémon)/Type (Energy and Trainer),Rule,Category,Previous stage,HP,Type,Weakness,Resistance (Type),Retreat,Move Name,Cost,Damage,Effect Explanation,Language
0,1,Basic {G} Energy,SVE,1,Basic Energy,NaN,NaN,NaN,NaN,{G},NaN,NaN,NaN,NaN,NaN,NaN,NaN,EN
1,2,Basic {R} Energy,SVE,2,Basic Energy,NaN,NaN,NaN,NaN,{R},NaN,NaN,NaN,NaN,NaN,NaN,NaN,EN
2,3,Basic {W} Energy,SVE,3,Basic Energy,NaN,NaN,NaN,NaN,{W},NaN,NaN,NaN,NaN,NaN,NaN,NaN,EN
3,4,Basic {L} Energy,SVE,4,Basic Energy,NaN,NaN,NaN,NaN,{L},NaN,NaN,NaN,NaN,NaN,NaN,NaN,EN
4,5,Basic {P} Energy,SVE,5,Basic Energy,NaN,NaN,NaN,NaN,{P},NaN,NaN,NaN,NaN,NaN,NaN,NaN,EN



JP Database


,カード ID,カード名,エキスパンションマーク,コレクション番号,ポケモンの進化の段階/エネルギー・トレーナーズの種類,ルール,カテゴリ,進化前,HP,タイプ,弱点,抵抗力,にげる,ワザ名,コスト,ダメージ,効果の説明,Language
0,1,基本【草】エネルギー,NaN,GRA,基本エネルギー,NaN,NaN,NaN,NaN,草,NaN,NaN,NaN,NaN,NaN,NaN,NaN,JP
1,2,基本【炎】エネルギー,NaN,FIR,基本エネルギー,NaN,NaN,NaN,NaN,炎,NaN,NaN,NaN,NaN,NaN,NaN,NaN,JP
2,3,基本【水】エネルギー,NaN,WAT,基本エネルギー,NaN,NaN,NaN,NaN,水,NaN,NaN,NaN,NaN,NaN,NaN,NaN,JP
3,4,基本【雷】エネルギー,NaN,LIG,基本エネルギー,NaN,NaN,NaN,NaN,雷,NaN,NaN,NaN,NaN,NaN,NaN,NaN,JP
4,5,基本【超】エネルギー,NaN,PSY,基本エネルギー,NaN,NaN,NaN,NaN,超,NaN,NaN,NaN,NaN,NaN,NaN,NaN,JP


# Deck Composition Analysis

In [3]:
from collections import Counter
from pathlib import Path
import pickle

import pandas as pd


DECK_PATH = Path("/kaggle/input/models/jek1wantaufik/buddy/other/pokemon/1/deck.pkl")

if not DECK_PATH.exists():
    DECK_PATH = Path("/kaggle/input/models/jek1wantaufik/buddy/other/pokemon/1/deck.pkl")


with open(DECK_PATH, "rb") as file:
    deck = pickle.load(file)


def find_column(dataframe: pd.DataFrame, keywords: list[str]) -> str:
    columns = {column.lower(): column for column in dataframe.columns}

    for keyword in keywords:
        for lower, original in columns.items():
            if keyword in lower:
                return original

    raise KeyError(f"Missing column containing {keywords}")


english_cards = card_database["EN"]

id_column = find_column(
    english_cards,
    ["cardid", "card_id", "id"],
)

name_column = find_column(
    english_cards,
    ["name"],
)

type_column = find_column(
    english_cards,
    ["type"],
)


lookup = (
    english_cards[[id_column, name_column, type_column]]
    .drop_duplicates(id_column)
    .set_index(id_column)
)


deck_summary = (
    pd.DataFrame(
        Counter(deck).items(),
        columns=["Card ID", "Copies"],
    )
    .sort_values("Copies", ascending=False)
)

deck_summary["Name"] = deck_summary["Card ID"].map(lookup[name_column])
deck_summary["Type"] = deck_summary["Card ID"].map(lookup[type_column])

deck_summary = deck_summary[
    ["Card ID", "Name", "Type", "Copies"]
].reset_index(drop=True)

display(deck_summary)

print(f"Deck Size : {sum(deck_summary['Copies'])}")
print(f"Unique Cards : {len(deck_summary)}")

,Card ID,Name,Type,Copies
0,6,Basic {F} Energy,Basic Energy,13
1,1152,Poké Pad,Item,4
2,1102,Dusk Ball,Item,4
3,678,Mega Lucario ex,Stage 1 Pokémon,4
4,1141,Premium Power Pro,Item,4
5,1142,Fighting Gong,Item,4
6,1192,Carmine,Supporter,4
7,1227,Lillie's Determination,Supporter,4
8,677,Riolu,Basic Pokémon,3
9,676,Solrock,Basic Pokémon,3


Deck Size : 60
Unique Cards : 17


# Card Role Analysis

In [4]:
ROLE_MAPPING = {
    "Makuhita": "Secondary Attacker",
    "Hariyama": "Secondary Attacker",
    "Riolu": "Primary Attacker",
    "Mega Lucario ex": "Primary Attacker",
    "Lunatone": "Energy Accelerator",
    "Solrock": "Support",
    "Switch": "Mobility",
    "Boss's Orders": "Board Control",
    "Hero's Cape": "Durability",
    "Premium Power Pro": "Draw Support",
    "Carmine": "Draw Support",
    "Lillie's Determination": "Draw Support",
    "Poké Pad": "Support Recovery",
    "Poké Ball": "Search",
    "Nest Ball": "Search",
    "Ultra Ball": "Search",
    "Dusk Ball": "Search",
    "Gravity Mountain": "Stadium",
    "Basic Fighting Energy": "Energy",
}


def assign_role(card_name: str) -> str:
    for keyword, role in ROLE_MAPPING.items():
        if keyword.lower() in card_name.lower():
            return role
    return "Other"


deck_roles = deck_summary.copy()

deck_roles["Role"] = deck_roles["Name"].apply(assign_role)

role_summary = (
    deck_roles
    .groupby("Role", as_index=False)["Copies"]
    .sum()
    .sort_values("Copies", ascending=False)
)

display(deck_roles)

display(role_summary)

,Card ID,Name,Type,Copies,Role
0,6,Basic {F} Energy,Basic Energy,13,Other
1,1152,Poké Pad,Item,4,Support Recovery
2,1102,Dusk Ball,Item,4,Search
3,678,Mega Lucario ex,Stage 1 Pokémon,4,Primary Attacker
4,1141,Premium Power Pro,Item,4,Draw Support
5,1142,Fighting Gong,Item,4,Other
6,1192,Carmine,Supporter,4,Draw Support
7,1227,Lillie's Determination,Supporter,4,Draw Support
8,677,Riolu,Basic Pokémon,3,Primary Attacker
9,676,Solrock,Basic Pokémon,3,Support


,Role,Copies
3,Other,20
0,Draw Support,12
4,Primary Attacker,7
9,Support Recovery,4
5,Search,4
6,Secondary Attacker,4
8,Support,3
1,Energy Accelerator,2
2,Mobility,2
7,Stadium,2


# Deck Philosophy

In [5]:
from textwrap import fill


def summarize_role(role: str, dataframe: pd.DataFrame) -> str:
    cards = dataframe.loc[dataframe["Role"] == role, "Name"].tolist()

    if not cards:
        return ""

    return f"{role}: {', '.join(cards)}."


def build_deck_philosophy(dataframe: pd.DataFrame) -> str:
    sections = [
        "The deck is designed around a heuristic-driven strategy that prioritizes efficient prize trading, stable board development, and consistent damage output.",
        summarize_role("Primary Attacker", dataframe),
        summarize_role("Secondary Attacker", dataframe),
        summarize_role("Energy Accelerator", dataframe),
        summarize_role("Search", dataframe),
        summarize_role("Draw Support", dataframe),
        summarize_role("Mobility", dataframe),
        summarize_role("Board Control", dataframe),
        summarize_role("Durability", dataframe),
        summarize_role("Stadium", dataframe),
        summarize_role("Energy", dataframe),
        "The overall objective is to establish attackers efficiently, maintain offensive pressure, and maximize decision quality through deterministic action selection.",
    ]

    return "\n\n".join(
        fill(section, width=100)
        for section in sections
        if section
    )


deck_philosophy = build_deck_philosophy(deck_roles)

print(deck_philosophy)

The deck is designed around a heuristic-driven strategy that prioritizes efficient prize trading,
stable board development, and consistent damage output.

Primary Attacker: Mega Lucario ex, Riolu.

Secondary Attacker: Makuhita, Hariyama.

Energy Accelerator: Lunatone.

Search: Dusk Ball.

Draw Support: Premium Power Pro, Carmine, Lillie's Determination.

Mobility: Switch.

Stadium: Gravity Mountain.

The overall objective is to establish attackers efficiently, maintain offensive pressure, and
maximize decision quality through deterministic action selection.


# Agent Architecture

In [6]:
from pathlib import Path
import ast
import pandas as pd


MAIN_PATH = Path("/kaggle/input/models/jek1wantaufik/buddy/other/pokemon/1/main.py")


source = MAIN_PATH.read_text(encoding="utf-8")
tree = ast.parse(source)


class AgentArchitecture(ast.NodeVisitor):
    def __init__(self):
        self.imports = []
        self.classes = []
        self.functions = []
        self.globals = []

    def visit_Import(self, node):
        for alias in node.names:
            self.imports.append(alias.name)

    def visit_ImportFrom(self, node):
        module = node.module or ""
        for alias in node.names:
            self.imports.append(f"{module}.{alias.name}")

    def visit_ClassDef(self, node):
        self.classes.append(node.name)
        self.generic_visit(node)

    def visit_FunctionDef(self, node):
        self.functions.append(node.name)
        self.generic_visit(node)

    def visit_Assign(self, node):
        if isinstance(node.parent, ast.Module):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    self.globals.append(target.id)

        self.generic_visit(node)


for parent in ast.walk(tree):
    for child in ast.iter_child_nodes(parent):
        child.parent = parent


architecture = AgentArchitecture()
architecture.visit(tree)


summary = pd.DataFrame(
    {
        "Component": [
            "Imports",
            "Global Variables",
            "Classes",
            "Functions",
        ],
        "Count": [
            len(architecture.imports),
            len(architecture.globals),
            len(architecture.classes),
            len(architecture.functions),
        ],
    }
)

display(summary)

display(
    pd.DataFrame(
        {
            "Function": architecture.functions
        }
    )
)

display(
    pd.DataFrame(
        {
            "Class": architecture.classes
        }
    )
)

,Component,Count
0,Imports,13
1,Global Variables,23
2,Classes,1
3,Functions,5


,Function
0,get_card
1,prize_count
2,pokemon_score
3,agent
4,energy_score


,Class
0,AttackPlan


# Decision Engine Analysis

In [7]:
import ast
import pandas as pd


class AgentAnalyzer(ast.NodeVisitor):
    def __init__(self):
        self.agent = None

    def visit_FunctionDef(self, node):
        if node.name == "agent":
            self.agent = node


analyzer = AgentAnalyzer()
analyzer.visit(tree)

agent_node = analyzer.agent


class DecisionAnalyzer(ast.NodeVisitor):
    def __init__(self):
        self.if_count = 0
        self.for_count = 0
        self.while_count = 0
        self.return_count = 0
        self.function_calls = set()

    def visit_If(self, node):
        self.if_count += 1
        self.generic_visit(node)

    def visit_For(self, node):
        self.for_count += 1
        self.generic_visit(node)

    def visit_While(self, node):
        self.while_count += 1
        self.generic_visit(node)

    def visit_Return(self, node):
        self.return_count += 1
        self.generic_visit(node)

    def visit_Call(self, node):
        if isinstance(node.func, ast.Name):
            self.function_calls.add(node.func.id)
        elif isinstance(node.func, ast.Attribute):
            self.function_calls.add(node.func.attr)
        self.generic_visit(node)


decision = DecisionAnalyzer()
decision.visit(agent_node)

metrics = pd.DataFrame(
    {
        "Metric": [
            "Conditional Branches",
            "For Loops",
            "While Loops",
            "Return Statements",
            "Unique Function Calls",
        ],
        "Value": [
            decision.if_count,
            decision.for_count,
            decision.while_count,
            decision.return_count,
            len(decision.function_calls),
        ],
    }
)

display(metrics)

display(
    pd.DataFrame(
        {
            "Function Call": sorted(decision.function_calls)
        }
    )
)

,Metric,Value
0,Conditional Branches,129
1,For Loops,12
2,While Loops,0
3,Return Statements,3
4,Unique Function Calls,14


,Function Call
0,AttackPlan
1,append
2,defaultdict
3,energy_score
4,enumerate
5,get_card
6,isinstance
7,len
8,min
9,pokemon_score


# Heuristic Scoring Analysis

In [8]:
import ast
import pandas as pd


class ScoreAnalyzer(ast.NodeVisitor):
    def __init__(self):
        self.assignments = 0
        self.augmentations = 0
        self.comparisons = 0

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id == "score":
                self.assignments += 1
        self.generic_visit(node)

    def visit_AugAssign(self, node):
        if isinstance(node.target, ast.Name) and node.target.id == "score":
            self.augmentations += 1
        self.generic_visit(node)

    def visit_Compare(self, node):
        names = {
            child.id
            for child in ast.walk(node)
            if isinstance(child, ast.Name)
        }

        if "score" in names:
            self.comparisons += 1

        self.generic_visit(node)


score_analyzer = ScoreAnalyzer()
score_analyzer.visit(agent_node)

score_summary = pd.DataFrame(
    {
        "Metric": [
            "Score Assignments",
            "Score Adjustments",
            "Score Comparisons",
        ],
        "Count": [
            score_analyzer.assignments,
            score_analyzer.augmentations,
            score_analyzer.comparisons,
        ],
    }
)

display(score_summary)

,Metric,Count
0,Score Assignments,36
1,Score Adjustments,45
2,Score Comparisons,1


# Attack Planning Analysis

In [9]:
import ast
import pandas as pd


class AttackPlanAnalyzer(ast.NodeVisitor):
    def __init__(self):
        self.attack_plan_fields = []
        self.plan_updates = []
        self.damage_variables = set()
        self.energy_variables = set()

    def visit_ClassDef(self, node):
        if node.name == "AttackPlan":
            for statement in node.body:
                if isinstance(statement, ast.Assign):
                    for target in statement.targets:
                        if isinstance(target, ast.Name):
                            self.attack_plan_fields.append(target.id)
        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Attribute):
                if (
                    isinstance(target.value, ast.Name)
                    and target.value.id == "plan"
                ):
                    self.plan_updates.append(target.attr)

            if isinstance(target, ast.Name):
                name = target.id.lower()
                if "damage" in name:
                    self.damage_variables.add(target.id)
                if "energy" in name:
                    self.energy_variables.add(target.id)

        self.generic_visit(node)


attack_analysis = AttackPlanAnalyzer()
attack_analysis.visit(tree)

attack_plan_summary = pd.DataFrame(
    {
        "Component": [
            "AttackPlan Fields",
            "Plan Updates",
            "Damage Variables",
            "Energy Variables",
        ],
        "Count": [
            len(attack_analysis.attack_plan_fields),
            len(set(attack_analysis.plan_updates)),
            len(attack_analysis.damage_variables),
            len(attack_analysis.energy_variables),
        ],
    }
)

display(attack_plan_summary)

display(
    pd.DataFrame(
        {
            "AttackPlan Field": attack_analysis.attack_plan_fields
        }
    )
)

display(
    pd.DataFrame(
        {
            "Updated Plan Attribute": sorted(set(attack_analysis.plan_updates))
        }
    )
)

,Component,Count
0,AttackPlan Fields,5
1,Plan Updates,5
2,Damage Variables,2
3,Energy Variables,5


,AttackPlan Field
0,attacker
1,target
2,attack_index
3,remain_hp
4,energy


,Updated Plan Attribute
0,attack_index
1,attacker
2,energy
3,remain_hp
4,target


# Heuristic Decision Categories

In [10]:
import ast
import re
import pandas as pd


KEYWORDS = {
    "Attack": [
        "attack",
        "damage",
        "remain_hp",
        "mega_brave",
    ],
    "Energy": [
        "energy",
        "attached",
    ],
    "Prize": [
        "prize",
    ],
    "Switch": [
        "switch",
        "retreat",
        "active",
        "bench",
    ],
    "Evolution": [
        "evolve",
        "stage",
    ],
    "Supporter": [
        "supporter",
        "boss",
        "carmine",
        "lillie",
    ],
    "Tool": [
        "tool",
        "cape",
    ],
    "Stadium": [
        "stadium",
        "gravity",
    ],
}


class HeuristicAnalyzer(ast.NodeVisitor):
    def __init__(self):
        self.categories = {key: 0 for key in KEYWORDS}

    def visit_If(self, node):
        expression = ast.unparse(node.test).lower()

        for category, keywords in KEYWORDS.items():
            if any(keyword in expression for keyword in keywords):
                self.categories[category] += 1

        self.generic_visit(node)


heuristic = HeuristicAnalyzer()
heuristic.visit(agent_node)

heuristic_summary = (
    pd.DataFrame(
        {
            "Category": heuristic.categories.keys(),
            "Decision Rules": heuristic.categories.values(),
        }
    )
    .sort_values("Decision Rules", ascending=False)
    .reset_index(drop=True)
)

display(heuristic_summary)

,Category,Decision Rules
0,Attack,18
1,Energy,15
2,Switch,11
3,Supporter,6
4,Prize,3
5,Evolution,3
6,Stadium,2
7,Tool,1


# Strategy Robustness Analysis

In [11]:
import ast
import pandas as pd


STATE_FEATURES = {
    "Hand": "hand",
    "Bench": "bench",
    "Active": "active",
    "Discard": "discard",
    "Prize": "prize",
    "Deck": "deck",
    "Stadium": "stadium",
    "Energy": "energy",
    "HP": "hp",
    "Weakness": "weakness",
    "Resistance": "resistance",
    "Supporter": "supporter",
    "Turn": "turn",
}


class StateFeatureAnalyzer(ast.NodeVisitor):
    def __init__(self):
        self.counts = {feature: 0 for feature in STATE_FEATURES}

    def visit_Name(self, node):
        name = node.id.lower()

        for feature, keyword in STATE_FEATURES.items():
            if keyword == name:
                self.counts[feature] += 1

    def visit_Attribute(self, node):
        name = node.attr.lower()

        for feature, keyword in STATE_FEATURES.items():
            if keyword == name:
                self.counts[feature] += 1

        self.generic_visit(node)


feature_analyzer = StateFeatureAnalyzer()
feature_analyzer.visit(agent_node)

feature_summary = (
    pd.DataFrame(
        {
            "Game State Feature": feature_analyzer.counts.keys(),
            "References": feature_analyzer.counts.values(),
        }
    )
    .sort_values("References", ascending=False)
    .reset_index(drop=True)
)

feature_summary["Used"] = feature_summary["References"] > 0

display(feature_summary)

coverage = feature_summary["Used"].mean()

print(f"State Feature Coverage: {coverage:.1%}")

,Game State Feature,References,Used
0,Active,7,True
1,Hand,5,True
2,Prize,5,True
3,Bench,4,True
4,HP,3,True
5,Turn,3,True
6,Energy,3,True
7,Discard,1,True
8,Stadium,1,True
9,Resistance,1,True


State Feature Coverage: 84.6%


# Strategy Evaluation Summary

In [12]:
import pandas as pd


evaluation = pd.DataFrame(
    {
        "Evaluation Criterion": [
            "Approach Clarity",
            "Technical Soundness",
            "Decision Consistency",
            "Strategy Robustness",
            "Deck Synergy",
            "Code Organization",
            "Interpretability",
        ],
        "Evidence": [
            f"{len(architecture.functions)} modular functions and {len(architecture.classes)} planner class.",
            f"{len(set(attack_analysis.plan_updates))} attack-planning attributes and {len(heuristic_summary)} heuristic categories.",
            f"{feature_summary['Used'].sum()} game-state features incorporated into decision making.",
            f"{coverage:.1%} state feature coverage across the decision engine.",
            f"{role_summary.shape[0]} strategic card roles identified in the deck.",
            "Deterministic rule-based implementation with modular helper functions.",
            "Explicit heuristic scoring enables transparent action selection.",
        ],
    }
)

display(evaluation)

,Evaluation Criterion,Evidence
0,Approach Clarity,5 modular functions and 1 planner class.
1,Technical Soundness,5 attack-planning attributes and 8 heuristic c...
2,Decision Consistency,11 game-state features incorporated into decis...
3,Strategy Robustness,84.6% state feature coverage across the decisi...
4,Deck Synergy,10 strategic card roles identified in the deck.
5,Code Organization,Deterministic rule-based implementation with m...
6,Interpretability,Explicit heuristic scoring enables transparent...


# Generate Strategy Report

In [13]:
from pathlib import Path


REPORT_PATH = Path("/kaggle/working/strategy_report.md")


report = f"""# Pokémon TCG AI Battle Challenge Strategy

## Executive Summary

This report presents a deterministic rule-based agent developed for the Pokémon TCG AI Battle competition.
The agent performs action planning by evaluating legal moves through heuristic scoring instead of machine learning
or reinforcement learning.

## Deck Overview

- Deck Size: {sum(deck_summary["Copies"])}
- Unique Cards: {len(deck_summary)}
- Strategic Roles: {len(role_summary)}

## Deck Philosophy

{deck_philosophy}

## Agent Architecture

| Metric | Value |
|--------|------:|
| Classes | {len(architecture.classes)} |
| Functions | {len(architecture.functions)} |
| Global Variables | {len(architecture.globals)} |

## Attack Planning

The agent constructs an attack plan before selecting actions.
The planner evaluates attackers, targets, attack options, prize trade,
damage potential, and energy requirements to maximize turn efficiency.

## Decision System

| Metric | Value |
|--------|------:|
| Conditional Branches | {decision.if_count} |
| For Loops | {decision.for_count} |
| Unique Function Calls | {len(decision.function_calls)} |

## Heuristic Categories

{heuristic_summary.to_markdown(index=False)}

## State Feature Coverage

Coverage: **{coverage:.1%}**

{feature_summary.to_markdown(index=False)}

## Strategy Evaluation

{evaluation.to_markdown(index=False)}

## Conclusion

The proposed agent follows a transparent heuristic-based planning framework.
Rather than relying on learned policies, it evaluates board states using
multiple strategic objectives including attack efficiency, prize trading,
energy management, board positioning, supporter timing, evolution, and
resource utilization.

The modular implementation improves interpretability while maintaining
consistent decision making across different board states.
"""

REPORT_PATH.write_text(report, encoding="utf-8")

print(REPORT_PATH)

print(report[:1200])

/kaggle/working/strategy_report.md
# Pokémon TCG AI Battle Challenge Strategy

## Executive Summary

This report presents a deterministic rule-based agent developed for the Pokémon TCG AI Battle competition.
The agent performs action planning by evaluating legal moves through heuristic scoring instead of machine learning
or reinforcement learning.

## Deck Overview

- Deck Size: 60
- Unique Cards: 17
- Strategic Roles: 10

## Deck Philosophy

The deck is designed around a heuristic-driven strategy that prioritizes efficient prize trading,
stable board development, and consistent damage output.

Primary Attacker: Mega Lucario ex, Riolu.

Secondary Attacker: Makuhita, Hariyama.

Energy Accelerator: Lunatone.

Search: Dusk Ball.

Draw Support: Premium Power Pro, Carmine, Lillie's Determination.

Mobility: Switch.

Stadium: Gravity Mountain.

The overall objective is to establish attackers efficiently, maintain offensive pressure, and
maximize decision quality through deterministic action 